# Liander Energy Data Analysis

This notebook demonstrates the data usage and analysis of energy data downloaded from [Liander Open Data](https://www.liander.nl/over-ons/open-data#lianderpower).

## Data Setup Instructions

To use this notebook, follow these steps:

1. **Download and Unzip**: Visit the [Liander Open Data website](https://www.liander.nl/over-ons/open-data#lianderpower), download the energy data (dataset name `LianderPower`) zip file, and unzip it. Inside the zip file you will find a `data` folder containing all the energy data files.

2. **Copy Data Folder**: Copy the extracted `data` folder to the root of this repository. Your directory structure should look like:
   ```
   liander-energy-data/
   ├── data/
   │   ├── (downloaded data files)
   │   └── ...
   ├── liander-energy-data.ipynb
   ├── main.py
   └── ...
   ```

3. **Review Documentation**: Read and understand the PDF documentation that accompanies the dataset to familiarize yourself with the data structure, columns, and any important notes.

## Import Libraries

In [ ]:
import polars as pl
import numpy as np
import plotly.express as px


## Load and Explore Data

In this section, we will load and explore the two parquet files:
- **measurements.parquet**: Contains energy measurement data
- **weather.parquet**: Contains weather information

We will read both files and display their structure, including the first 5 rows and column information.

In [13]:
from pathlib import Path

# Define a function to load and explore a parquet file
def load_and_explore_parquet(file_path):
    """Load parquet file and print header and column information"""
    print(f"\n{'='*60}")
    print(f"File: {Path(file_path).name}")
    print('='*60)
    
    # Load the parquet file
    df = pl.read_parquet(file_path)
    
    # Print head (first 5 rows)
    print(f"\nFirst 5 rows:")
    print(df.head(5))
    
    # Print column information in a nice table
    print(f"\nColumn Information:")
    schema_df = pl.DataFrame({
        "Column Name": df.columns,
        "Data Type": [str(dtype) for dtype in df.schema.values()]
    })
    with pl.Config(tbl_rows=15):
        print(schema_df)
    
    return df

# Load the two parquet files
df_measurements = load_and_explore_parquet("data/measurements.parquet")
df_weather = load_and_explore_parquet("data/weather.parquet")


File: measurements.parquet

First 5 rows:
shape: (5, 11)
┌────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│ locati ┆ locati ┆ lat    ┆ lon    ┆ sensor ┆ measur ┆ weathe ┆ start_ ┆ sample ┆ num_va ┆ values │
│ on_id  ┆ on_nam ┆ ---    ┆ ---    ┆ _unit  ┆ ement_ ┆ r_loca ┆ time   ┆ _inter ┆ lues   ┆ ---    │
│ ---    ┆ e      ┆ f64    ┆ f64    ┆ ---    ┆ type   ┆ tion_i ┆ ---    ┆ val_s  ┆ ---    ┆ list[f │
│ i64    ┆ ---    ┆        ┆        ┆ str    ┆ ---    ┆ d      ┆ i64    ┆ ---    ┆ i32    ┆ 32]    │
│        ┆ str    ┆        ┆        ┆        ┆ str    ┆ ---    ┆        ┆ i32    ┆        ┆        │
│        ┆        ┆        ┆        ┆        ┆        ┆ i64    ┆        ┆        ┆        ┆        │
╞════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╡
│ 0      ┆ pseudo ┆ 51.780 ┆ 5.1185 ┆ P      ┆ pseudo ┆ 100000 ┆ 137877 ┆ 300    ┆ 104860 ┆ [5.52, │
│        ┆ _aggre ┆ 99     ┆ 1   

## Data Structure Notes

### Join Key
The **weather_location_id** column is the key to join the two dataframes (`df_measurements` and `df_weather`). This ID links measurements to their corresponding weather data for analysis.

### Meta-Frame Structure
Both dataframes follow a meta-frame structure:
- The **values** column contains a **list of floats** for each row
- Each row represents a location/time point, and the `values` column contains the entire time series of data for that point
- This means each row is not a single value, but rather a compressed representation of a complete time series
- To perform time-series analysis, you may need to explode or unnest the `values` column to work with individual data points

## Constructing Time Series Data for a Location

In this section, we will construct a new dataframe for a specific location_id by combining measurements and weather data:

### Process:
1. **Filter by Location**: Extract all rows from `df_measurements` that match a specific `location_id`

2. **Cross-Reference Weather Data**: For each measurement row, use the `weather_location_id` to find the corresponding rows in `df_weather` that contain the relevant weather data

3. **Expand to Time Series**: For each row in both dataframes, reconstruct the full time series by:
   - Using the `start_time` as the first timestamp
   - Using the `sample_interval` to generate uniform timestamps for each subsequent value
   - Using the `num_values` to determine the length of the time series
   - Combining these with the `values` column (list of floats) to create complete time series

4. **Combine into Single Dataframe**: Merge all time series into one dataframe by:
   - Aligning timestamps across all time series
   - Using linear interpolation/extrapolation to fill values at different time intervals
   - **Retaining NaNs** where data is missing (do not fill NaN values)
   - Creating a unified dataset with timestamps and all measurement and weather columns

In [25]:
location_name = "pseudo_aggregate_00017"

In [31]:
def construct_location_timeseries(location_name, meas, weather):
    """
    Construct a combined time series dataframe for a specific location.
    
    Parameters:
    - location_name: The location name to filter
    - meas: The measurements dataframe
    - weather: The weather dataframe
    
    Returns:
    - A combined dataframe with aligned timestamps and linearly interpolated values
    """
    
    # Step 1: Filter measurements for the location
    meas_filtered = meas.filter(pl.col("location_name") == location_name)
    
    if meas_filtered.height == 0:
        raise ValueError(f"No measurements found for location_name: {location_name}")
    
    # Step 2: Get weather_location_id(s) and cross-reference weather data
    weather_location_ids = meas_filtered["weather_location_id"].unique().to_list()
    weather_filtered = weather.filter(pl.col("weather_location_id").is_in(weather_location_ids))
    
    if weather_filtered.height == 0:
        raise ValueError(f"No weather data found for location_name: {location_name}")
    
    # Step 3: Expand time series for measurements and weather
    timeseries_list = []
    
    # Expand measurements
    for row in meas_filtered.iter_rows(named=True):
        start_time = row["start_time"]
        sample_interval = row["sample_interval_s"]
        values = row["values"]
        num_values = row["num_values"]
        sensor_unit = row["sensor_unit"]
        
        # Generate timestamps using Polars datetime_range
        ts_df = pl.DataFrame({
            "timestamp": pl.datetime_range(
                start=pl.datetime(second=start_time, time_unit="s"),
                periods=num_values,
                interval=f"{sample_interval}s",
                eager=True
            ),
            sensor_unit: values,
        })
        timeseries_list.append(ts_df)
    
    # Expand weather
    for row in weather_filtered.iter_rows(named=True):
        start_time = row["start_time"]
        sample_interval = row["sample_interval_s"]
        values = row["values"]
        num_values = row["num_values"]
        feature_name = row["feature_name"]
        
        # Generate timestamps using Polars datetime_range
        ts_df = pl.DataFrame({
            "timestamp": pl.datetime_range(
                start=pl.datetime(second=start_time, time_unit="s"),
                periods=num_values,
                interval=f"{sample_interval}s",
                eager=True
            ),
            feature_name: values,
        })
        timeseries_list.append(ts_df)
    
    # Step 4: Combine all time series with interpolation
    # Start with the first dataframe
    combined_ts = timeseries_list[0]
    
    # Join remaining dataframes on timestamp
    for ts_df in timeseries_list[1:]:
        combined_ts = combined_ts.join(ts_df, on="timestamp", how="outer")
    
    # Sort by timestamp
    combined_ts = combined_ts.sort("timestamp").set_index("timestamp")
    
    # Linear interpolation for non-NaN values, retaining original NaNs
    # Apply interpolation to all columns except timestamp
    value_cols = [col for col in combined_ts.columns if col != "timestamp"]
    combined_ts = combined_ts.with_columns([
        pl.col(col).interpolate(method="linear") for col in value_cols
    ])
    
    return combined_ts

In [32]:
df_combined = construct_location_timeseries(location_name=location_name, meas=df_measurements, weather=df_weather)

TypeError: datetime_() missing 3 required positional arguments: 'year', 'month', and 'day'

In [28]:
df_measurements.filter(pl.col("location_name") == location_name)

location_id,location_name,lat,lon,sensor_unit,measurement_type,weather_location_id,start_time,sample_interval_s,num_values,values
i64,str,f64,f64,str,str,i64,i64,i32,i32,list[f32]
17,"""pseudo_aggregate_00017""",51.846539,5.288327,"""P""","""pseudo_aggregate""",1000013,1378771200,300,1048608,"[6.9, 6.87, … 8.87]"
